# Solutions — groupby, Named Aggregations, Pivots, RFM, Rolling Windows

Exercise notebook: `09_pandas/04_groupby_aggregation.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Simple grouped sum

**Working solution**

In [ ]:
def revenue_by(df, column):
    return df.loc[df["status"] != "CANCELLED"].groupby(column)["amount"].sum().round(2).sort_values(ascending=False)

**Explanation**

Filter → group → select column → reduce → sort. Selecting `['amount']` *before* the reduction avoids aggregating every column.

**Complexity:** O(n).

**Common mistakes**

- `df.groupby(col).sum()['amount']` — sums all numeric columns first (slow, and errors on strings in pandas 2+).

## Exercise 2 — 🟡 Medium · Named aggregations

**Working solution**

In [ ]:
def summary_by(df, keys):
    out = (df.groupby(keys, as_index=False)
             .agg(orders=("order_id", "nunique"), units=("quantity", "sum"), revenue=("amount", "sum"), customers=("customer_id", "nunique")))
    out["avg_order_value"] = (out["revenue"] / out["orders"]).round(2)
    return out[[*keys, "orders", "units", "revenue", "avg_order_value", "customers"]].sort_values(keys).reset_index(drop=True)

**Explanation**

Named aggregation gives you clean column names in one step (no MultiIndex columns to flatten). Ratios are computed *after* aggregation from the aggregated parts.

**Complexity:** O(n).

**Common mistakes**

- `agg({'amount': ['sum', 'mean']})` → MultiIndex columns that are awkward downstream.
- Averaging row-level amounts when you mean revenue per order.

## Exercise 3 — 🟡 Medium · Pivot tables

**Working solution**

In [ ]:
import pandas as pd

def daily_channel_pivot(df):
    valid = df.loc[df["status"] != "CANCELLED"].assign(order_date=lambda d: d["order_date"].dt.strftime("%Y-%m-%d"))
    pivot = pd.pivot_table(valid, index="order_date", columns="channel", values="amount", aggfunc="sum", fill_value=0)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    pivot.columns.name = None
    return pivot.assign(total=pivot.sum(axis=1))

**Explanation**

`pivot_table` = group by two keys + unstack. `fill_value=0` handles empty cells; `sum(axis=1)` is a row-wise total (vectorized across columns).

**Complexity:** O(n).

**Common mistakes**

- `df.pivot()` (no aggregation) → `ValueError` on duplicate index/column pairs.

## Exercise 4 — 🟠 Hard · RFM customer segmentation

**Working solution**

In [ ]:
import pandas as pd

def rfm(df, as_of):
    valid = df.loc[df["status"] != "CANCELLED"]
    g = valid.groupby("customer_id").agg(last=("order_date", "max"), frequency=("order_id", "nunique"), monetary=("amount", "sum"))
    g["recency_days"] = (pd.Timestamp(as_of) - g["last"]).dt.days
    def score(series, reverse=False):
        ranked = series.rank(method="first", ascending=not reverse)
        return pd.qcut(ranked, 3, labels=[1, 2, 3]).astype(int)
    g["r"] = score(g["recency_days"], reverse=True)
    g["f"] = score(g["frequency"])
    g["m"] = score(g["monetary"])
    g["rfm_score"] = g["r"].astype(str) + g["f"].astype(str) + g["m"].astype(str)
    return g[["recency_days", "frequency", "monetary", "r", "f", "m", "rfm_score"]].sort_index()

**Explanation**

Ranking first (`method="first"`) makes `qcut` robust to ties (otherwise duplicate bin edges raise). Reversing the rank direction for recency encodes 'more recent is better'.

**Complexity:** O(n + c log c).

**Common mistakes**

- `qcut` directly on values with many ties → `ValueError: Bin edges must be unique`.

## Challenge — 🔴 Very Hard · Daily metrics with rolling windows and shares

**Working solution**

In [ ]:
import pandas as pd

def daily_metrics(df):
    valid = df.loc[df["status"] != "CANCELLED"]
    daily = valid.groupby(["country", "order_date"])["amount"].sum()
    dates = pd.date_range(valid["order_date"].min(), valid["order_date"].max(), freq="D", name="order_date")
    full_index = pd.MultiIndex.from_product([sorted(valid["country"].unique()), dates], names=["country", "order_date"])
    out = daily.reindex(full_index, fill_value=0.0).rename("revenue").reset_index()
    by_country = out.groupby("country")["revenue"]
    out["rolling_7d"] = by_country.transform(lambda s: s.rolling(7, min_periods=1).mean())
    out["dod_change"] = by_country.diff()
    day_total = out.groupby("order_date")["revenue"].transform("sum")
    out["share_of_day"] = (out["revenue"] / day_total).where(day_total > 0, 0.0).round(4)
    return out.sort_values(["country", "order_date"]).reset_index(drop=True)

**Explanation**

`reindex` onto a full date × country grid makes missing days explicit zeros — otherwise rolling windows silently span gaps. `transform` returns a result aligned to the original
rows, so group-level values (rolling means, daily totals) can be used in row-level arithmetic.

**Complexity:** O(dates × countries).

**Common mistakes**

- Rolling over rows instead of days when days are missing.
- Computing shares with a merge back instead of `transform`.